# Data visualization and preprocessing

In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
import sys
sys.path.append("..")

PROJECT_ROOT = Path.cwd().resolve().parent   
sys.path.insert(0, str(PROJECT_ROOT)) 


from src.datasets.build_dataset import build_cohort_b_dataset
from src.datasets.cleaning import filter_physiological_outliers, preprocess_glucose, extract_eligible_blocks
from src.visualization.eda_plots import plot_participant_boxplots, plot_participant_mean_cv, inter_individual_spread, plot_glucose_evol_day, plot_daily_mean, plot_hour_day_heatmap, plot_input_vs_target_mean
from src.datasets.data_splitting import split_and_save_cohort
from src.datasets.features import split_input_target, compute_cgm_features, compute_targets_features, build_xy
from src.datasets.glucose_audit import load_cohort_b, check_timezone, check_outliers, check_units, check_gaps, FIG

## Data loading

CGM recordings of participants from the same cohort, with times and dates of recording

In [ ]:
build_cohort_b_dataset()
DATA_DIR = Path("../../data/processed")
data = pd.read_csv(DATA_DIR / "cohort_b_glucose_data.csv")

## Glucose audit - data quality

Before proceeding with data preprocessing, we perform a quality audit of the raw Cohort B glucose data.

The audit checks four aspects:
- **Timezone:** determine whether timestamps are in UTC or already in Swiss time.
- **Outliers:** identify suspicious glucose values without removing them.
- **Units:** check whether glucose measurements are expressed in mmol/L.
- **Gaps:** analyze missing periods between consecutive glucose readings.

The audit functions are imported from `src/datasets/glucose_audit.py` to avoid duplicating code.

In [4]:
df = load_cohort_b()

FIG.mkdir(parents=True, exist_ok=True)

check_timezone(df)
check_outliers(df)
check_units(df)
check_gaps(df)


== 1. Timezone: hour of the night trough ==
  raw hours in the file       : 3 h
  converted to Swiss time     : 5 h
  winter (raw hours)          : 4 h
  summer (raw hours)          : 2 h

== 2. Outliers ==
  min / max value                 : 1.43 / 27.80
  missing or <= 0                 : 0
  below 2.0                       : 3
  above 27.8                     : 0
  above 25.0                     : 194
  flat runs of 12+ identical readings : 211

== 3. Units ==
  participant means, min / max          : 3.62 / 11.62
  participants with a mean above 30     : 0 (likely mg/dL)

== 4. Gaps (> 30 min) ==
  number of gaps: 4541
ts
30 min-1 h    1441
1-2 h         1495
2-6 h         1225
6-24 h         238
over 24 h      142
  participants with a gap over 24 h : 99
  longest gap, median / max         : 2.3 h / 18888.9 h


The results of these checks will help us make informed decisions during preprocessing.
No data is removed or modified during this audit. The identified issues will be considered when defining the cleaning and preprocessing steps.

## Glucose distribution visualization

In [ ]:
stats = inter_individual_spread(data)
print (stats)

In [ ]:
plot_participant_boxplots(data)

In [ ]:
plot_participant_mean_cv(data)

## Preprocessing

remove outliers and missing values, deal with different timestamps, and select eligible participants

In [ ]:
data_good_time = preprocess_glucose("cohort_b_glucose_data.csv")
data_clean = filter_physiological_outliers(data_good_time)
data_preprocess, _, _ = extract_eligible_blocks(data_clean)

### Glucose evolution visualization

In [ ]:
plot_glucose_evol_day(data_preprocess)

In [ ]:
plot_daily_mean(data_preprocess)

In [ ]:
plot_hour_day_heatmap(data_preprocess)

## Split train and test data

In [ ]:
train_data, val_data, test_data = split_and_save_cohort(data_preprocess)

## Compute glucose data features for different durations

K = 1 day

In [ ]:
input_df_1d, target_df = split_input_target(train_data, k_days=1)
features_1d = compute_cgm_features(input_df_1d, night_hours=(0, 6), day_hours=(8, 20))
target = compute_targets_features (target_df)


K = 3 days

In [ ]:
input_df_3d, target_df = split_input_target(train_data, k_days=3)
features_3d = compute_cgm_features(input_df_3d, night_hours=(0, 6), day_hours=(8, 20))

K = 5 days

In [ ]:
input_df_5d, target_df = split_input_target(train_data, k_days=5)
features_5d = compute_cgm_features(input_df_5d, night_hours=(0, 6), day_hours=(8, 20))

K = 7 days

In [ ]:
input_df_7d, target_df = split_input_target(train_data, k_days=7)
features_7d = compute_cgm_features(input_df_7d, night_hours=(0, 6), day_hours=(8, 20))

In [ ]:
plot_input_vs_target_mean(train_data)